In [ ]:
from pathlib import Path
from textwrap import dedent
import json
import os
from datetime import datetime

In [ ]:
class MetaRegistro(type):
    cantidad_instancias = 0

    def __call__(cls, *args, **kwargs):
        MetaRegistro.cantidad_instancias += 1
        return super().__call__(*args, **kwargs)

In [ ]:
def registrar(funcion):
    def envoltura(*args, **kwargs):
        print(f"[INFO] Ejecutando: {funcion.__name__}")
        return funcion(*args, **kwargs)
    return envoltura

In [ ]:
class Entidad(metaclass=MetaRegistro):

    def mostrar(self):
        print("Entidad")


class Libro(Entidad):

    def __init__(self, titulo, autor, isbn, año, paginas):
        self.titulo = titulo
        self.autor = autor
        self.isbn = isbn
        self.año = año
        self.paginas = paginas

    def mostrar(self):
        print(f"{self.titulo} - {self.autor}")


class Usuario(Entidad):

    def __init__(self, nombre, apellido, dni, correo):
        self.nombre = nombre
        self.apellido = apellido
        self.dni = dni
        self.correo = correo

    def mostrar(self):
        print(f"{self.nombre} {self.apellido}")

In [ ]:
class Prestamo:

    def __init__(self, libro, usuario):
        self.libro = libro
        self.usuario = usuario
        self.fecha_prestamo = str(datetime.now().date())
        self.fecha_devolucion = None

    def devolver(self):
        self.fecha_devolucion = str(datetime.now().date())


In [ ]:
class Biblioteca:

    def __init__(self):

        self.libros = []
        self.usuarios = []
        self.prestamos = []

        self.cargar_datos()

    def guardar_datos(self):

        datos = {
            "libros": [],
            "usuarios": [],
            "prestamos": []
        }

        for libro in self.libros:
            datos["libros"].append({
                "titulo": libro.titulo,
                "autor": libro.autor,
                "isbn": libro.isbn,
                "anio": libro.anio,
                "paginas": libro.paginas
            })

        for usuario in self.usuarios:
            datos["usuarios"].append({
                "nombre": usuario.nombre,
                "apellido": usuario.apellido,
                "dni": usuario.dni,
                "correo": usuario.correo
            })

        for prestamo in self.prestamos:
            datos["prestamos"].append({
                "isbn": prestamo.libro.isbn,
                "dni": prestamo.usuario.dni,
                "fecha_prestamo": prestamo.fecha_prestamo,
                "fecha_devolucion": prestamo.fecha_devolucion
            })

        with open("biblioteca.json", "w", encoding="utf8") as archivo:
            json.dump(datos, archivo, indent=4)

    def cargar_datos(self):

        if not os.path.exists("biblioteca.json"):
            return

        with open("biblioteca.json", "r", encoding="utf8") as archivo:
            datos = json.load(archivo)

        for l in datos["libros"]:
            self.libros.append(
                Libro(
                    l["titulo"],
                    l["autor"],
                    l["isbn"],
                    l["anio"],
                    l["paginas"]
                )
            )

        for u in datos["usuarios"]:
            self.usuarios.append(
                Usuario(
                    u["nombre"],
                    u["apellido"],
                    u["dni"],
                    u["correo"]
                )
            )

        for p in datos["prestamos"]:

            libro = self.buscar_libro(p["isbn"])
            usuario = self.buscar_usuario(p["dni"])

            if libro and usuario:
                prestamo = Prestamo(libro, usuario)
                prestamo.fecha_prestamo = p["fecha_prestamo"]
                prestamo.fecha_devolucion = p["fecha_devolucion"]
                self.prestamos.append(prestamo)

    @registrar
    def agregar_libro(self, libro):
        self.libros.append(libro)
        self.guardar_datos()

    @registrar
    def agregar_usuario(self, usuario):
        self.usuarios.append(usuario)
        self.guardar_datos()

    def listar_libros(self):

        if len(self.libros) == 0:
            print("No hay libros")
            return

        for libro in self.libros:
            libro.mostrar()

    def listar_usuarios(self):

        if len(self.usuarios) == 0:
            print("No hay usuarios")
            return

        for usuario in self.usuarios:
            usuario.mostrar()

    def buscar_libro(self, isbn):

        for libro in self.libros:
            if libro.isbn == isbn:
                return libro

        return None

    def buscar_usuario(self, dni):

        for usuario in self.usuarios:
            if usuario.dni == dni:
                return usuario

        return None

    @registrar
    def eliminar_libro(self, isbn):

        libro = self.buscar_libro(isbn)

        if libro:
            self.libros.remove(libro)
            self.guardar_datos()

    @registrar
    def eliminar_usuario(self, dni):

        usuario = self.buscar_usuario(dni)

        if usuario:
            self.usuarios.remove(usuario)
            self.guardar_datos()

    @registrar
    def prestar_libro(self, isbn, dni):

        libro = self.buscar_libro(isbn)
        usuario = self.buscar_usuario(dni)

        if not libro:
            print("Libro inexistente")
            return

        if not usuario:
            print("Usuario inexistente")
            return

        for prestamo in self.prestamos:
            if prestamo.libro.isbn == isbn and prestamo.fecha_devolucion is None:
                print("Libro ya prestado")
                return

        self.prestamos.append(Prestamo(libro, usuario))
        self.guardar_datos()

    @registrar
    def devolver_libro(self, isbn):

        for prestamo in self.prestamos:

            if prestamo.libro.isbn == isbn and prestamo.fecha_devolucion is None:
                prestamo.devolver()
                self.guardar_datos()
                return

    def prestamos_activos(self):

        for prestamo in self.prestamos:

            if prestamo.fecha_devolucion is None:

                print(
                    prestamo.libro.titulo,
                    "-",
                    prestamo.usuario.nombre,
                    "-",
                    prestamo.fecha_prestamo
                )


def menu():

    biblioteca = Biblioteca()

    while True:

        print("===== BIBLIOTECA DIGITAL =====")
        print("1. Agregar libro")
        print("2. Listar libros")
        print("3. Eliminar libro")
        print("4. Agregar usuario")
        print("5. Listar usuarios")
        print("6. Eliminar usuario")
        print("7. Prestar libro")
        print("8. Devolver libro")
        print("9. Prestamos activos")
        print("0. Salir")

        opcion = input("Opcion: ")

        if opcion == "1":

            biblioteca.agregar_libro(
                Libro(
                    input("Titulo: "),
                    input("Autor: "),
                    input("ISBN: "),
                    int(input("Año: ")),
                    int(input("Paginas: "))
                )
            )

        elif opcion == "2":
            biblioteca.listar_libros()

        elif opcion == "3":
            biblioteca.eliminar_libro(input("ISBN: "))

        elif opcion == "4":

            biblioteca.agregar_usuario(
                Usuario(
                    input("Nombre: "),
                    input("Apellido: "),
                    input("DNI: "),
                    input("Correo: ")
                )
            )

        elif opcion == "5":
            biblioteca.listar_usuarios()

        elif opcion == "6":
            biblioteca.eliminar_usuario(input("DNI: "))

        elif opcion == "7":
            biblioteca.prestar_libro(
                input("ISBN: "),
                input("DNI: ")
            )

        elif opcion == "8":
            biblioteca.devolver_libro(input("ISBN: "))

        elif opcion == "9":
            biblioteca.prestamos_activos()

        elif opcion == "0":
            break

In [ ]:
def test_crear_libro():

    libro = Libro(
        "Python",
        "Juan Perez",
        "123",
        2024,
        300
    )

    assert libro.titulo == "Python"
    assert libro.autor == "Juan Perez"
    assert libro.isbn == "123"
    assert libro.paginas == 300


def test_crear_usuario():

    usuario = Usuario(
        "Ana",
        "Gomez",
        "11111111",
        "ana@gmail.com"
    )

    assert usuario.nombre == "Ana"
    assert usuario.apellido == "Gomez"
    assert usuario.dni == "11111111"


def test_agregar_libro():

    biblioteca = Biblioteca()

    cantidad_inicial = len(biblioteca.libros)

    libro = Libro(
        "Libro Test",
        "Autor Test",
        "ISBN001",
        2024,
        150
    )

    biblioteca.agregar_libro(libro)

    assert len(biblioteca.libros) == cantidad_inicial + 1


def test_agregar_usuario():

    biblioteca = Biblioteca()

    cantidad_inicial = len(biblioteca.usuarios)

    usuario = Usuario(
        "Carlos",
        "Lopez",
        "22222222",
        "carlos@gmail.com"
    )

    biblioteca.agregar_usuario(usuario)

    assert len(biblioteca.usuarios) == cantidad_inicial + 1


def test_buscar_libro():

    biblioteca = Biblioteca()

    libro = Libro(
        "Busqueda",
        "Autor",
        "ISBNBUSCAR",
        2024,
        200
    )

    biblioteca.agregar_libro(libro)

    encontrado = biblioteca.buscar_libro("ISBNBUSCAR")

    assert encontrado == libro


def test_buscar_usuario():

    biblioteca = Biblioteca()

    usuario = Usuario(
        "Pedro",
        "Perez",
        "33333333",
        "pedro@gmail.com"
    )

    biblioteca.agregar_usuario(usuario)

    encontrado = biblioteca.buscar_usuario("33333333")

    assert encontrado == usuario


def test_prestar_libro():

    biblioteca = Biblioteca()

    libro = Libro(
        "Prestamo",
        "Autor",
        "ISBNPRESTAMO",
        2024,
        100
    )

    usuario = Usuario(
        "Lucia",
        "Diaz",
        "44444444",
        "lucia@gmail.com"
    )

    biblioteca.agregar_libro(libro)
    biblioteca.agregar_usuario(usuario)

    cantidad_inicial = len(biblioteca.prestamos)

    biblioteca.prestar_libro(
        "ISBNPRESTAMO",
        "44444444"
    )

    assert len(biblioteca.prestamos) == cantidad_inicial + 1


def test_devolver_libro():

    biblioteca = Biblioteca()

    libro = Libro(
        "Devolucion",
        "Autor",
        "ISBNDEV",
        2024,
        250
    )

    usuario = Usuario(
        "Maria",
        "Suarez",
        "55555555",
        "maria@gmail.com"
    )

    biblioteca.agregar_libro(libro)
    biblioteca.agregar_usuario(usuario)

    biblioteca.prestar_libro(
        "ISBNDEV",
        "55555555"
    )

    biblioteca.devolver_libro("ISBNDEV")

    assert biblioteca.prestamos[0].fecha_devolucion is not None


def test_no_prestar_dos_veces():

    biblioteca = Biblioteca()

    libro = Libro(
        "Duplicado",
        "Autor",
        "ISBNDUP",
        2024,
        100
    )

    usuario = Usuario(
        "Jose",
        "Garcia",
        "66666666",
        "jose@gmail.com"
    )

    biblioteca.agregar_libro(libro)
    biblioteca.agregar_usuario(usuario)

    biblioteca.prestar_libro(
        "ISBNDUP",
        "66666666"
    )

    biblioteca.prestar_libro(
        "ISBNDUP",
        "66666666"
    )

    assert len(biblioteca.prestamos) == 1


def test_metaclase():

    cantidad_inicial = MetaRegistro.cantidad_instancias

    Libro(
        "Meta",
        "Autor",
        "ISBNMETA",
        2024,
        100
    )

    assert MetaRegistro.cantidad_instancias == cantidad_inicial + 1


def test_polimorfismo():

    objetos = [
        Libro("A", "B", "1", 2024, 100),
        Usuario("Ana", "Perez", "123", "ana@gmail.com")
    ]

    for objeto in objetos:
        assert hasattr(objeto, "mostrar")


def test_guardado_json():

    biblioteca = Biblioteca()

    cantidad_inicial = len(biblioteca.libros)

    biblioteca.agregar_libro(
        Libro(
            "Persistencia",
            "Autor",
            "ISBNJSON",
            2024,
            100
        )
    )

    nueva_biblioteca = Biblioteca()

    assert len(nueva_biblioteca.libros) >= cantidad_inicial + 1

In [ ]:
if __name__ == "__main__":
    menu()

===== BIBLIOTECA DIGITAL =====
1. Agregar libro
2. Listar libros
3. Eliminar libro
4. Agregar usuario
5. Listar usuarios
6. Eliminar usuario
7. Prestar libro
8. Devolver libro
9. Prestamos activos
0. Salir
Opcion: 0
